In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report

In [2]:
import string
import pickle

In [3]:
import nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

In [6]:

# Load dataset
df = pd.read_csv("final_dataset.csv")

In [7]:
df['label'].unique()

array([0, 1, 2])

In [8]:
df.describe()

,label
count,515345.000000
mean,0.822172
std,0.692418
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,2.000000


In [10]:
df['label'].value_counts()

label
1    251970
0    177509
2     85866
Name: count, dtype: int64

In [ ]:


# # Split data
# X_train, X_test, y_train, y_test = train_test_split(
#     df['text'], df['label'], test_size=0.2
# )

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)




# TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Model
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced')
model.fit(X_train, y_train)

# Accuracy
print("Accuracy:", model.score(X_test, y_test))

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

Accuracy: 0.9592311946366027
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     35502
           1       0.99      0.94      0.97     50394
           2       0.82      0.97      0.89     17173

    accuracy                           0.96    103069
   macro avg       0.94      0.96      0.95    103069
weighted avg       0.96      0.96      0.96    103069



update if needded below 

In [ ]:
# vectorizer = TfidfVectorizer(
#     ngram_range=(1,2),
#     max_features=5000
# )

# model = LogisticRegression(
#     max_iter=1000,
#     class_weight='balanced'
# )

In [12]:
def preprocess_text(text):
    text = text.lower()  # lowercase
    
    # remove punctuation
    text = "".join([char for char in text if char not in string.punctuation])
    
    # remove stopwords
    words = text.split()
    words = [word for word in words if word not in stopwords.words('english')]
    
    return " ".join(words)

In [13]:
label_map = {
    0: "Normal Message",
    1: "Spam Message",
    2: "Phishing Message"
}

def predict(text):
    text = preprocess_text(text)
    text_vector = vectorizer.transform([text])
    prediction = model.predict(text_vector)[0]
    
    return label_map[prediction]

In [14]:
print(predict("Congratulations! You won a free iPhone"))
print(predict("Let's meet tomorrow at 5 PM"))
print(predict("Verify your bank account password immediately"))

Spam Message
Normal Message
Spam Message


In [15]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

MODEL BUILD SUCCESSFULLY